# Mise en production, indexation avec Molmo2-8B

Projet TorneH, fonds Éric Jantzen du musée d'Orsay. Ce notebook est la version de mise en production du test Molmo2 du benchmark. Il applique la même détection à vocabulaire contrôlé à une liste de photographies, et non plus à une image à la fois. Il a été testé sur un échantillon de 8 photographies.

- Entrée, un fichier Excel listant les photographies à traiter avec le nom de fichier, l'ID photo et l'ID bâtiment, le dossier images à plat et le thésaurus anglais au format json
- Traitement, une seule étape par photographie, la détection, sans localisation ni image en sortie, afin de limiter le temps de calcul
- Sortie, un fichier raw_detection.jsonl avec une ligne par photographie contenant l'ID photo, l'ID bâtiment, le nom du fichier et la réponse brute du modèle, ou le message d'erreur
- Robustesse, écriture de chaque résultat au fil du traitement et reprise après interruption en ignorant les ID photo déjà traités
- Suite du traitement, le rattachement des termes aux TermID est réalisé dans un second script
- Environnement, Google Colab sur GPU T4, modèle en quantification 4 bit, mode échantillon réglé par SAMPLE_SIZE, à 8 pour le test et à None pour traiter toute la liste

Les fichiers à placer dans un dossier molmo_batch du Google Drive sont le dossier images, le fichier images_de_reference_renomme.xlsx et le fichier thesau_angl.json.

## 1. Installation

In [ ]:
# Installation des bibliothèques. Fixation des versions de transformers et de pillow, issues des corrections de compatibilité de la phase de test. Ces versions ne doivent pas être modifiées.
!pip uninstall -y transformers -q
!pip install -q transformers==4.57.1 accelerate torch torchvision einops "pillow==11.1.0" bitsandbytes openpyxl pandas

## Redémarrage de la session requis avant la suite

## 2. Vérification après le redémarrage

Cette cellule est à exécuter seule, juste après le redémarrage.

In [ ]:
# Vérification de la version de transformers et de la disponibilité du GPU.
import transformers, torch
assert transformers.__version__ == "4.57.1", transformers.__version__
print("transformers OK :", transformers.__version__)
print("CUDA disponible :", torch.cuda.is_available())

## 3. Montage du Google Drive

In [ ]:
# Montage du Google Drive, qui contient les photographies et reçoit les résultats.
from google.colab import drive
drive.mount('/content/drive')

## 4. Script A, détection sur un échantillon de test

Le paramètre SAMPLE_SIZE vaut 8 en mode test. Après validation, il est à passer à None pour traiter l'ensemble des photographies.

In [ ]:
# Script A. Détection des termes du thésaurus sur les photographies du fonds, à partir de la liste de l'Excel.
# Seule la réponse brute du modèle est sauvegardée, dans un fichier jsonl comportant une ligne par photographie. Le rattachement aux TermID est réalisé dans un second script.
# Le mode échantillon est actif, voir SAMPLE_SIZE.
import json
import re
import time
import traceback
from pathlib import Path

import pandas as pd
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch

# Paramètres. Chemins sur le Drive, noms des colonnes de l'Excel, thésaurus et modèle.
IMAGES_DIR = Path("/content/drive/MyDrive/molmo_batch/images")   # dossier des 17000 images (en vrac)
EXCEL_PATH = Path("/content/drive/MyDrive/molmo_batch/images_de_reference_renomme.xlsx")
EXCEL_COL_FICHIER = "Nom fichier image"
EXCEL_COL_ID_PHOTO = "ID photo"
EXCEL_COL_ID_BATIMENT = "ID bâtiment"

THESAURUS_PATH = Path("/content/drive/MyDrive/molmo_batch/thesau_angl.json")
TERM_FIELD = "Term_angl"
MODEL_ID = "allenai/Molmo2-8B"
MAX_NEW_TOKENS = 200

RESULTS_PATH = Path("/content/drive/MyDrive/molmo_batch/raw_detection.jsonl")

# SAMPLE_SIZE à None traite l'ensemble de la liste, un nombre active le mode échantillon de test. La seed rend l'échantillon reproductible.
SAMPLE_SIZE = 8            # None = traite toute la liste ; un nombre = échantillon de test
SAMPLE_RANDOM_SEED = 42    # fixe pour que l'échantillon soit reproductible

# Chargement de la liste des photographies à traiter. Les lignes sans nom de fichier ou sans identifiant sont écartées. En mode échantillon, un tirage aléatoire est effectué.
df = pd.read_excel(EXCEL_PATH)
df = df.dropna(subset=[EXCEL_COL_FICHIER, EXCEL_COL_ID_PHOTO])
print(f"{len(df)} lignes valides dans l'Excel.", flush=True)

if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=SAMPLE_RANDOM_SEED)
    print(f"Mode échantillon : {len(df)} photos sélectionnées.", flush=True)

# Chargement du thésaurus et construction du prompt de détection. Le prompt intègre la liste complète et impose de ne répondre qu'avec ces termes.
with open(THESAURUS_PATH, "r", encoding="utf-8") as f:
    thesaurus_raw = json.load(f)

words = []
if isinstance(thesaurus_raw, list):
    for fiche in thesaurus_raw:
        if isinstance(fiche, dict) and TERM_FIELD in fiche and isinstance(fiche[TERM_FIELD], str):
            term = fiche[TERM_FIELD].strip()
            if term:
                words.append(term)

words = sorted(set(words))
print(f"{len(words)} termes chargés depuis le thésaurus.", flush=True)

word_list_str = ", ".join(words)
detection_prompt = (
    "Here is a controlled list of keywords (thesaurus): "
    f"[{word_list_str}].\n\n"
    "Carefully analyze the provided image and identify ONLY the elements "
    "from this list that are actually visible in the image.\n"
    "Strict constraints:\n"
    "- Only use words present in the list, copied exactly as given.\n"
    "- Do not invent any word outside the list.\n"
    "- Do not give any explanation, no sentences.\n"
    "- Answer only as a comma-separated list."
)

# Reprise après interruption. Les photographies déjà présentes dans le fichier de résultats sont écartées, ce qui permet de relancer le notebook après une coupure de session.
already_done = set()
if RESULTS_PATH.exists():
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                already_done.add(row["id_photo"])
            except (json.JSONDecodeError, KeyError):
                continue
    print(f"{len(already_done)} photos déjà traitées (reprise).", flush=True)

df = df[~df[EXCEL_COL_ID_PHOTO].isin(already_done)]
print(f"{len(df)} photos restant à traiter.", flush=True)

if len(df) == 0:
    print("Rien à faire, tout est déjà traité.", flush=True)
    raise SystemExit(0)

# Chargement du modèle en quantification 4 bit pour tenir dans la mémoire du GPU.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["vision_backbone"],
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=torch.bfloat16, device_map="auto"
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=quant_config,
    dtype=torch.bfloat16,
    device_map="auto",
)
print("Modèle chargé.", flush=True)


def ask_molmo(prompt_text, image):
    """Envoie un prompt et une image à Molmo2 et renvoie la réponse sous forme de texte."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {"type": "image", "image": image},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    generated_tokens = generated_ids[0, inputs["input_ids"].size(1):]
    return processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)


# Boucle de traitement. Chaque résultat est écrit dans le fichier dès qu'il est obtenu, pour éviter toute perte en cas d'interruption. Les erreurs sont consignées dans le fichier sans interrompre le traitement. Le temps moyen par photographie est affiché pour estimer la durée du traitement complet.
start_time = time.time()
n_done_this_run = 0

with open(RESULTS_PATH, "a", encoding="utf-8") as out_f:
    for _, row in df.iterrows():
        fichier = str(row[EXCEL_COL_FICHIER]).strip()
        id_photo = row[EXCEL_COL_ID_PHOTO]
        id_batiment = row.get(EXCEL_COL_ID_BATIMENT, None)

        image_path = IMAGES_DIR / fichier

        try:
            if not image_path.exists():
                raise FileNotFoundError(f"Fichier introuvable dans {IMAGES_DIR} : {fichier}")

            image = Image.open(image_path).convert("RGB")
            raw_detection = ask_molmo(detection_prompt, image)

            result = {
                "id_photo": id_photo,
                "id_batiment": id_batiment,
                "fichier": fichier,
                "raw_detection": raw_detection,
            }
            out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
            out_f.flush()

        except Exception as e:
            error_result = {
                "id_photo": id_photo,
                "fichier": fichier,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
            out_f.write(json.dumps(error_result, ensure_ascii=False) + "\n")
            out_f.flush()
            print(f"[ERREUR] {fichier} (id_photo={id_photo}) : {e}", flush=True)

        n_done_this_run += 1
        elapsed = time.time() - start_time
        avg = elapsed / n_done_this_run
        print(f"[{n_done_this_run}/{len(df)}] {fichier} — {avg:.1f}s/photo en moyenne", flush=True)

print(f"\nTerminé. {n_done_this_run} photos traitées dans cette session.", flush=True)
print(f"Résultats cumulés dans : {RESULTS_PATH}", flush=True)

## 5. Vérification du résultat

Affichage des dernières lignes du fichier de résultats pour contrôle.

In [ ]:
# Relecture du fichier de résultats et affichage des huit dernières lignes.
import json

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    lines = [json.loads(l) for l in f if l.strip()]

for row in lines[-8:]:
    print("---")
    print("id_photo:", row.get("id_photo"))
    print("fichier:", row.get("fichier"))
    if "error" in row:
        print("ERREUR:", row["error"])
    else:
        print("raw_detection:", row.get("raw_detection"))